In [1]:
# Setup
import os, sys, torch
from argparse import Namespace

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
sys.path.append(repo_root) if repo_root not in sys.path else None

from revlm.config_utils import configure_args
from revlm import VQAModel, VQADataset
from revlm.editors.auto_q import BiasLayer


In [2]:
# Load model & dataset
model_name = "llava"  # Options: "blip", "llava", "qwen3", "qwen3_4b"

args = Namespace(config="revlm/config/config.yaml", editor="reasonedit", model_name=model_name,
                 dataset_name="aokvqa", task="mc", batch_size=1, split="all",
                 rationale=False, cot=False, subsample=100, overwrite=False)
args.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
config = configure_args(args, config_path=args.config)

model = VQAModel(config)
dataset = VQADataset(config)
print(f"Model: {config.model.name}, Dataset: {len(dataset)} samples")


In [3]:
# Initialize BiasLayer and get candidate layers
bias = BiasLayer(config, model, pool_method="mean")
layers = bias.get_candidate_layers()


In [4]:
# Compute bias at each layer (OPTIMIZED: batched forward passes)
scores = bias.compute(dataset, layers, n_samples=10)


In [5]:
# Plot bias scores
bias.plot(scores)


In [6]:
# Cleanup
bias.cleanup()
del model
torch.cuda.empty_cache()


# Load Saved Results (After Running Jobs)

After running `jobs/bias_layer/run.sh`, use the cells below to load aggregated results with error bars.


In [7]:
# Setup for loading saved results
import os, sys, torch
from argparse import Namespace

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
sys.path.append(repo_root) if repo_root not in sys.path else None

from revlm.config_utils import configure_args
from revlm.editors.auto_q import BiasLayer


In [10]:
# Load aggregated results from saved runs (with error bars)
model_name = "qwen3_4b"  # Options: "blip", "llava", "qwen3", "qwen3_4b"
pool_method = "mean"     # Options: "mean", "last"

args = Namespace(config="revlm/config/config.yaml", editor="reasonedit", model_name=model_name,
                 dataset_name="aokvqa", task="mc", batch_size=1, split="all",
                 rationale=False, cot=False)
args.device = torch.device("cpu")
config = configure_args(args, config_path=args.config)

# Create BiasLayer just for loading (no model needed)
bias_loader = BiasLayer.__new__(BiasLayer)
bias_loader.config = config
bias_loader.device = args.device
bias_loader.pool_method = pool_method
# Load aggregated results
agg_scores = bias_loader.load_results_k(out_dir="results/auto_q_ckpts/bias_layer")
if agg_scores:
    print(f"Loaded {len(agg_scores)} layers")


In [11]:
# Plot aggregated results with error bars
if agg_scores:
    bias_loader._classify_layers = lambda layers: BiasLayer._classify_layers(bias_loader, layers)
    bias_loader.plot = lambda scores, **kwargs: BiasLayer.plot(bias_loader, scores, **kwargs)
    bias_loader.plot(agg_scores)
else:
    print("No saved results found. Run jobs/bias_layer/run.sh first.")
